# Import Libraries

In [7]:
import sys
print(sys.version)

3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]


In [8]:
#!pip install textblob==0.17.1
from textblob import TextBlob

In [9]:
# Data manipulation
#!pip install vaderSentiment
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Text processing
import re
import nltk
from nltk.corpus import stopwords

# Lexicons
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob

# Evaluation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nltk.download('stopwords')

KeyboardInterrupt: 

# Load Dataset

In [ ]:

file_path = file_path = "Gift_Cards.jsonl"
df = pd.read_json(file_path, lines=True)

df = pd.read_json(file_path, lines=True)

# Basic Information

In [ ]:
print("Total Reviews:", len(df))
print("Average Rating:", df['overall'].mean())

df.info()
df.describe()

# Rating Distribution

In [ ]:
sns.countplot(x='overall', data=df)
plt.title("Distribution of Ratings")
plt.show()

# Reviews Per Product

In [ ]:
reviews_per_product = df.groupby('asin').size()

print(reviews_per_product.describe())

plt.hist(reviews_per_product, bins=30)
plt.title("Distribution of Reviews Per Product")
plt.show()

# Reviews Per User

In [ ]:
reviews_per_user = df.groupby('reviewerID').size()

print(reviews_per_user.describe())

plt.hist(reviews_per_user, bins=30)
plt.title("Distribution of Reviews Per User")
plt.show()

# Review Length Analysis

In [ ]:
df['review_length'] = df['reviewText'].str.len()

print(df['review_length'].describe())

plt.hist(df['review_length'], bins=50)
plt.title("Review Length Distribution")
plt.show()

# Check Duplicates

In [ ]:
print("Duplicate Rows:", df.select_dtypes(exclude=['object']).duplicated().sum())



## TEXT PRE-PROCESSING

### Label Sentiment

In [ ]:
def label_sentiment(rating):
    if rating >= 4:
        return "Positive"
    elif rating == 3:
        return "Neutral"
    else:
        return "Negative"

df['sentiment'] = df['overall'].apply(label_sentiment)

df['sentiment'].value_counts()

# Clean Text

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df['cleaned_text'] = df['reviewText'].astype(str).apply(clean_text)

df[['reviewText', 'cleaned_text']].head()

 # Select 1000 Random Reviews

In [ ]:
df_sample = df.sample(n=1000, random_state=42)

df_sample.shape

# MODEL 1 — VADER


In [ ]:
analyzer = SentimentIntensityAnalyzer()

def vader_sentiment(text):
    score = analyzer.polarity_scores(text)['compound']
    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    else:
        return "Neutral"

df_sample['vader_prediction'] = df_sample['cleaned_text'].apply(vader_sentiment)

df_sample[['sentiment','vader_prediction']].head()

# MODEL 2 — TextBlob

In [ ]:
def textblob_sentiment(text):
    score = TextBlob(text).sentiment.polarity
    if score > 0:
        return "Positive"
    elif score < 0:
        return "Negative"
    else:
        return "Neutral"

df_sample['textblob_prediction'] = df_sample['cleaned_text'].apply(textblob_sentiment)

df_sample[['sentiment','textblob_prediction']].head()

# Accuracy

In [ ]:
print("VADER Accuracy:", accuracy_score(df_sample['sentiment'], df_sample['vader_prediction']))
print("TextBlob Accuracy:", accuracy_score(df_sample['sentiment'], df_sample['textblob_prediction']))

# Classification Reports

In [ ]:
print("VADER REPORT")
print(classification_report(df_sample['sentiment'], df_sample['vader_prediction']))

print("TEXTBLOB REPORT")
print(classification_report(df_sample['sentiment'], df_sample['textblob_prediction']))

# Confusion Matrix

In [ ]:
print("VADER Confusion Matrix")
print(confusion_matrix(df_sample['sentiment'], df_sample['vader_prediction']))

print("TEXTBLOB Confusion Matrix")
print(confusion_matrix(df_sample['sentiment'], df_sample['textblob_prediction']))